# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import random
import string
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BOS_ID = 0
VOCAB = list(string.ascii_uppercase)
char2id = {c: i + 1 for i, c in enumerate(VOCAB)}
id2char = {i + 1: c for i, c in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB) + 1  # + BOS


## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
def random_example(seqlen=10):
    s = ''.join(random.choice(VOCAB) for _ in range(seqlen))
    rev = s[::-1]
    enc = [char2id[c] for c in s]
    dec = [BOS_ID] + [char2id[c] for c in rev[:-1]]
    y = [char2id[c] for c in rev]
    return s, enc, dec, y


def generate_batch(batch_size=64, seqlen=10):
    examples, enc_x, dec_x, y = [], [], [], []
    for _ in range(batch_size):
        s, e, d, t = random_example(seqlen)
        examples.append(s)
        enc_x.append(e)
        dec_x.append(d)
        y.append(t)
    return examples, torch.tensor(enc_x, dtype=torch.long), torch.tensor(dec_x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

print(generate_batch(2, 10)[0])


['UDAXIHHEXD', 'VXRCSNBACG']


# 建立sequence to sequence 模型

In [3]:
class mySeq2SeqModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, emb_dim=64, hidden=128):
        super().__init__()
        self.hidden = hidden
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.GRU(emb_dim, hidden, batch_first=True)
        self.decoder = nn.GRU(emb_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, enc_x, dec_x):
        enc_emb = self.embed(enc_x)
        _, h = self.encoder(enc_emb)            # h: [1,B,H]
        dec_emb = self.embed(dec_x)
        dec_out, _ = self.decoder(dec_emb, h)   # [B,T,H]
        logits = self.fc(dec_out)               # [B,T,V]
        return logits

    @torch.no_grad()
    def predict(self, enc_x, steps=10):
        self.eval()
        enc_emb = self.embed(enc_x)
        _, h = self.encoder(enc_emb)
        bsz = enc_x.size(0)
        cur = torch.full((bsz, 1), BOS_ID, dtype=torch.long, device=enc_x.device)
        outs = []
        for _ in range(steps):
            dec_emb = self.embed(cur)
            dec_out, h = self.decoder(dec_emb, h)
            logits = self.fc(dec_out[:, -1, :])
            cur = torch.argmax(logits, dim=1, keepdim=True)
            outs.append(cur)
        return torch.cat(outs, dim=1)


# Loss函数以及训练逻辑

In [4]:
def compute_loss(logits, labels):
    # logits [B,T,V], labels [B,T]
    return F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))


def train_one_step(model, optimizer, enc_x, dec_x, y):
    model.train()
    optimizer.zero_grad()
    logits = model(enc_x, dec_x)
    loss = compute_loss(logits, y)
    loss.backward()
    optimizer.step()
    pred = torch.argmax(logits, dim=-1)
    acc = (pred == y).float().mean()
    return loss.item(), acc.item()


def train(model, optimizer, seqlen=20, steps=1000, batch_size=128, print_every=100):
    model.to(device)
    for i in range(1, steps + 1):
        _, enc_x, dec_x, y = generate_batch(batch_size, seqlen)
        enc_x, dec_x, y = enc_x.to(device), dec_x.to(device), y.to(device)
        loss, acc = train_one_step(model, optimizer, enc_x, dec_x, y)
        if i % print_every == 0:
            print(f'step {i}: loss={loss:.4f}, acc={acc:.4f}')


# 训练迭代

In [5]:
model = mySeq2SeqModel()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
train(model, optimizer, seqlen=20, steps=1200, batch_size=128, print_every=120)


step 120: loss=2.9983, acc=0.1324
step 240: loss=2.4974, acc=0.2680
step 360: loss=2.2521, acc=0.3184
step 480: loss=2.0747, acc=0.3652
step 600: loss=1.9358, acc=0.3875
step 720: loss=1.8429, acc=0.4230
step 840: loss=1.7488, acc=0.4391
step 960: loss=1.6728, acc=0.4660
step 1080: loss=1.5719, acc=0.4684
step 1200: loss=1.5639, acc=0.4781


# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def decode_ids(ids):
    return ''.join(id2char.get(int(i), '?') for i in ids)


def sequence_reversal(n=10, seqlen=10):
    model.eval()
    examples, enc_x, _, y = generate_batch(n, seqlen)
    enc_x = enc_x.to(device)
    pred = model.predict(enc_x, steps=seqlen).cpu().numpy()
    y = y.numpy()
    for i in range(n):
        print('src :', examples[i])
        print('gold:', decode_ids(y[i]))
        print('pred:', decode_ids(pred[i]))
        print('-' * 32)

sequence_reversal(n=10, seqlen=10)


src : LKNBRGHHMA
gold: AMHHGRBNKL
pred: AHMMHRNGVL
--------------------------------
src : VHMDFZOCSZ
gold: ZSCOZFDMHV
pred: ZSZOCDFMVH
--------------------------------
src : QHNISDQTTA
gold: ATTQDSINHQ
pred: ATTDQNACVH
--------------------------------
src : YYURTUOEKZ
gold: ZKEOUTRUYY
pred: ZKUEOYTURV
--------------------------------
src : GFQKHIWGSS
gold: SSGWIHKQFG
pred: SSGWGRSOMD
--------------------------------
src : ZBNFGAOMRB
gold: BRMOAGFNBZ
pred: BRMOBJMNZD
--------------------------------
src : EHTMIZLOVY
gold: YVOLZIMTHE
pred: VYOMTIPHND
--------------------------------
src : DUJJALASZS
gold: SZSALAJJUD
pred: SSASJBPMDJ
--------------------------------
src : ZVHJGBJOWC
gold: CWOJBGJHVZ
pred: WCOJGZBJVH
--------------------------------
src : WETOUCVNDT
gold: TDNVCUOTEW
pred: TDNVCUNEWV
--------------------------------
